# Notebook 04: Sets & Sorted Sets

## Part 1: Sets

Redis Sets are **unordered collections of unique strings**. Just like Python's `set()`:
- **No duplicates** — adding the same value twice has no effect
- **O(1) add/remove/check** — instant lookup
- **Set operations** — union, intersection, difference

**Common uses:** Tags, unique visitors, online users, mutual friends.

In [ ]:
import redis

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
### SADD — Add Members to a Set

In [ ]:
# Redis CLI: SADD fruits "apple" "banana" "cherry" "apple"
added = r.sadd('fruits', 'apple', 'banana', 'cherry', 'apple')  # 'apple' is duplicate
print(f"Added {added} NEW members (duplicate ignored)")  # 3, not 4

# SMEMBERS — Get all members
# Redis CLI: SMEMBERS fruits
print(f"Set contents: {r.smembers('fruits')}")
# Note: order is NOT guaranteed!

### SISMEMBER, SCARD, SREM

In [ ]:
# SISMEMBER — Check if value is in the set (O(1) — super fast!)
# Redis CLI: SISMEMBER fruits "apple"
print(f"Is 'apple' in fruits? {r.sismember('fruits', 'apple')}")   # True
print(f"Is 'mango' in fruits? {r.sismember('fruits', 'mango')}")   # False

# SCARD — Get the number of members (cardinality)
# Redis CLI: SCARD fruits
print(f"Number of fruits: {r.scard('fruits')}")

# SREM — Remove members
# Redis CLI: SREM fruits "banana"
r.srem('fruits', 'banana')
print(f"After removing banana: {r.smembers('fruits')}")

### SPOP and SRANDMEMBER — Random Elements

In [ ]:
r.delete('cards')
r.sadd('cards', 'Ace', 'King', 'Queen', 'Jack', '10', '9', '8', '7')

# SRANDMEMBER — Get random member WITHOUT removing
# Redis CLI: SRANDMEMBER cards
random_card = r.srandmember('cards')
print(f"Random card (still in set): {random_card}")
print(f"Set size after SRANDMEMBER: {r.scard('cards')}")

# Get 3 random members
three_cards = r.srandmember('cards', 3)
print(f"3 random cards: {three_cards}")

# SPOP — Remove and return random member
# Redis CLI: SPOP cards
drawn = r.spop('cards')
print(f"\nDrawn card (removed): {drawn}")
print(f"Set size after SPOP: {r.scard('cards')}")

---
### Set Operations — Where Sets Really Shine!

```
   Set A            Set B
 ┌───────┐       ┌───────┐
 │ apple │       │ banana│
 │ banana│───────│ cherry│
 │ mango │       │ grape │
 └───────┘       └───────┘

 UNION:        {apple, banana, mango, cherry, grape}  (everything)
 INTERSECTION: {banana}                                (common to both)
 DIFF (A-B):   {apple, mango}                         (in A but not B)
 DIFF (B-A):   {cherry, grape}                        (in B but not A)
```

In [ ]:
r.delete('setA', 'setB')
r.sadd('setA', 'apple', 'banana', 'mango')
r.sadd('setB', 'banana', 'cherry', 'grape')

print(f"Set A: {r.smembers('setA')}")
print(f"Set B: {r.smembers('setB')}")

# SUNION — Everything in A or B (or both)
print(f"\nUNION (A | B):        {r.sunion('setA', 'setB')}")

# SINTER — Only elements in BOTH A and B
print(f"INTERSECTION (A & B): {r.sinter('setA', 'setB')}")

# SDIFF — Elements in A but NOT in B
print(f"DIFF (A - B):         {r.sdiff('setA', 'setB')}")
print(f"DIFF (B - A):         {r.sdiff('setB', 'setA')}")

In [ ]:
# You can also STORE the results in a new set
# Redis CLI: SINTERSTORE common setA setB
r.sinterstore('common', 'setA', 'setB')
print(f"Stored intersection: {r.smembers('common')}")

---
### Real-World: Tag System

In [ ]:
# Tag articles with topics — find articles with common tags
r.sadd('article:1:tags', 'python', 'redis', 'backend')
r.sadd('article:2:tags', 'python', 'django', 'backend')
r.sadd('article:3:tags', 'javascript', 'react', 'frontend')

print(f"Article 1 tags: {r.smembers('article:1:tags')}")
print(f"Article 2 tags: {r.smembers('article:2:tags')}")

# Common tags between articles 1 and 2
common = r.sinter('article:1:tags', 'article:2:tags')
print(f"\nCommon tags (1 & 2): {common}")

# All unique tags across all articles
all_tags = r.sunion('article:1:tags', 'article:2:tags', 'article:3:tags')
print(f"All tags: {all_tags}")

### Real-World: Unique Visitors

In [ ]:
# Track unique visitors — duplicates are automatically ignored!
r.delete('visitors:homepage')

visitors = ['user_1', 'user_2', 'user_3', 'user_1', 'user_2', 'user_4', 'user_1']
for v in visitors:
    r.sadd('visitors:homepage', v)

print(f"Total visits: {len(visitors)}")
print(f"Unique visitors: {r.scard('visitors:homepage')}")
print(f"Who visited: {r.smembers('visitors:homepage')}")

### Real-World: Mutual Friends

In [ ]:
r.sadd('friends:alice', 'bob', 'charlie', 'diana', 'eve')
r.sadd('friends:bob', 'alice', 'charlie', 'frank', 'eve')

mutual = r.sinter('friends:alice', 'friends:bob')
print(f"Alice's friends: {r.smembers('friends:alice')}")
print(f"Bob's friends:   {r.smembers('friends:bob')}")
print(f"Mutual friends:  {mutual}")

# People Alice knows but Bob doesn't (friend suggestions for Bob!)
suggestions = r.sdiff('friends:alice', 'friends:bob')
print(f"Friend suggestions for Bob: {suggestions}")

---
## Part 2: Sorted Sets (ZSets)

Sorted Sets are like Sets, but each member has a **score** (a floating-point number). Members are automatically sorted by score.

- **Unique members** (like sets)
- **Ordered by score** (lowest to highest by default)
- **O(log n) add/remove** — very fast even with millions of members

**Think of it as:** A leaderboard where each player has a score.

```
Sorted Set: leaderboard
┌────────────┬───────┐
│ Member     │ Score │
├────────────┼───────┤
│ bob        │  85   │
│ charlie    │  92   │
│ alice      │ 100   │
└────────────┴───────┘
  (sorted by score ascending)
```

### ZADD — Add Members with Scores

In [ ]:
# Redis CLI: ZADD leaderboard 100 "alice" 85 "bob" 92 "charlie"
# In Python, we pass a dict: {member: score}
r.zadd('leaderboard', {'alice': 100, 'bob': 85, 'charlie': 92, 'diana': 78, 'eve': 95})

# ZRANGE — Get members sorted by score (low to high)
# Redis CLI: ZRANGE leaderboard 0 -1 WITHSCORES
print("Leaderboard (low to high):")
for member, score in r.zrange('leaderboard', 0, -1, withscores=True):
    print(f"  {member}: {score}")

### ZREVRANGE — Get Members High to Low (Leaderboard Order)

In [ ]:
# Redis CLI: ZREVRANGE leaderboard 0 -1 WITHSCORES
print("Leaderboard (high to low):")
for rank, (member, score) in enumerate(r.zrevrange('leaderboard', 0, -1, withscores=True), 1):
    print(f"  #{rank} {member}: {int(score)} pts")

### ZSCORE, ZRANK, ZREVRANK

In [ ]:
# ZSCORE — Get the score of a member
# Redis CLI: ZSCORE leaderboard "alice"
print(f"Alice's score: {r.zscore('leaderboard', 'alice')}")

# ZRANK — Get rank (0-based, lowest score = rank 0)
# Redis CLI: ZRANK leaderboard "alice"
print(f"Alice's rank (low→high): {r.zrank('leaderboard', 'alice')}")

# ZREVRANK — Rank from highest score (0 = #1)
# Redis CLI: ZREVRANK leaderboard "alice"
print(f"Alice's rank (high→low): {r.zrevrank('leaderboard', 'alice')}")
print(f"  (0 means #1 — alice has the highest score!)")

### ZINCRBY — Update Scores

In [ ]:
# Redis CLI: ZINCRBY leaderboard 15 "bob"
print(f"Bob's score before: {r.zscore('leaderboard', 'bob')}")

new_score = r.zincrby('leaderboard', 15, 'bob')  # +15 points
print(f"Bob's score after +15: {new_score}")

# Check new rankings
print("\nUpdated leaderboard:")
for rank, (member, score) in enumerate(r.zrevrange('leaderboard', 0, -1, withscores=True), 1):
    print(f"  #{rank} {member}: {int(score)} pts")

### ZRANGEBYSCORE — Query by Score Range

In [ ]:
# Get members with scores between 90 and 100
# Redis CLI: ZRANGEBYSCORE leaderboard 90 100 WITHSCORES
top_scorers = r.zrangebyscore('leaderboard', 90, 100, withscores=True)
print("Players scoring 90-100:")
for member, score in top_scorers:
    print(f"  {member}: {int(score)}")

# ZCOUNT — Count members in score range
count = r.zcount('leaderboard', 90, 100)
print(f"\nNumber of 90+ scorers: {count}")

### ZREM, ZCARD

In [ ]:
# ZCARD — Count total members
print(f"Total players: {r.zcard('leaderboard')}")

# ZREM — Remove a member
# Redis CLI: ZREM leaderboard "diana"
r.zrem('leaderboard', 'diana')
print(f"After removing diana: {r.zcard('leaderboard')} players")

### ZUNIONSTORE / ZINTERSTORE — Combine Sorted Sets

In [ ]:
# Two games — combine scores!
r.zadd('game1:scores', {'alice': 100, 'bob': 80, 'charlie': 90})
r.zadd('game2:scores', {'alice': 85, 'bob': 95, 'diana': 70})

# ZUNIONSTORE — Sum scores across both games
# Redis CLI: ZUNIONSTORE total:scores 2 game1:scores game2:scores AGGREGATE SUM
r.zunionstore('total:scores', ['game1:scores', 'game2:scores'], aggregate='SUM')

print("Combined scores (SUM):")
for member, score in r.zrevrange('total:scores', 0, -1, withscores=True):
    print(f"  {member}: {int(score)}")

# ZINTERSTORE with MAX — Only players in BOTH games, take highest score
r.zinterstore('best:scores', ['game1:scores', 'game2:scores'], aggregate='MAX')

print("\nBest score per player (only in both games):")
for member, score in r.zrevrange('best:scores', 0, -1, withscores=True):
    print(f"  {member}: {int(score)}")

---
### Real-World: Complete Game Leaderboard

In [ ]:
import random

r.delete('game:leaderboard')

# Simulate 20 players with random scores
players = [f'player_{i}' for i in range(1, 21)]
for player in players:
    score = random.randint(0, 1000)
    r.zadd('game:leaderboard', {player: score})

# Top 5 players
print("=== TOP 5 PLAYERS ===")
for rank, (player, score) in enumerate(r.zrevrange('game:leaderboard', 0, 4, withscores=True), 1):
    print(f"  #{rank} {player}: {int(score)} pts")

# Specific player lookup
p = 'player_7'
print(f"\n{p}'s rank: #{r.zrevrank('game:leaderboard', p) + 1}")
print(f"{p}'s score: {int(r.zscore('game:leaderboard', p))}")

# Players scoring above 500
high_scorers = r.zrangebyscore('game:leaderboard', 500, '+inf')
print(f"\nPlayers with 500+ points: {len(high_scorers)}")

# Total players
print(f"Total players: {r.zcard('game:leaderboard')}")

### Real-World: Priority Queue

In [ ]:
# Use scores as priority (lower = higher priority)
r.delete('priority_queue')

r.zadd('priority_queue', {
    'critical_bug_fix': 1,       # Highest priority
    'security_patch': 2,
    'feature_request': 5,
    'ui_improvement': 8,
    'documentation': 10          # Lowest priority
})

print("Tasks by priority (highest first):")
for task, priority in r.zrange('priority_queue', 0, -1, withscores=True):
    print(f"  [{int(priority)}] {task}")

# Pop highest priority task
task = r.zpopmin('priority_queue')
print(f"\nProcessing: {task[0][0]} (priority {int(task[0][1])})")
print(f"Remaining: {r.zcard('priority_queue')} tasks")

### Real-World: Time-Series (Recent Items by Timestamp)

In [ ]:
import time

r.delete('recent:logins')

# Use timestamp as score for time-ordered data
logins = ['alice', 'bob', 'charlie', 'diana', 'eve']
for user in logins:
    r.zadd('recent:logins', {user: time.time()})
    time.sleep(0.1)  # Small delay so timestamps differ

# Get most recent logins (newest first)
print("Recent logins (newest first):")
for user, ts in r.zrevrange('recent:logins', 0, -1, withscores=True):
    print(f"  {user} at {ts:.2f}")

---
## Sets vs Sorted Sets

| Feature | Set | Sorted Set |
|---|---|---|
| Ordering | None | Sorted by score |
| Unique members | Yes | Yes |
| Scores | No | Yes (float) |
| Add complexity | O(1) | O(log n) |
| Set operations | SUNION, SINTER, SDIFF | ZUNIONSTORE, ZINTERSTORE |
| Best for | Tags, unique tracking | Leaderboards, priority queues |

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

### Sets
```
SADD key member         → Add member
SMEMBERS key            → Get all members
SISMEMBER key member    → Check membership (O(1)!)
SCARD key               → Count members
SREM key member         → Remove member
SPOP key                → Remove random member
SUNION key1 key2        → Union of sets
SINTER key1 key2        → Intersection of sets
SDIFF key1 key2         → Difference of sets
```

### Sorted Sets
```
ZADD key {member: score}   → Add with score
ZRANGE key 0 -1            → Get by rank (low→high)
ZREVRANGE key 0 -1         → Get by rank (high→low)
ZSCORE key member          → Get score
ZRANK/ZREVRANK key member  → Get rank
ZINCRBY key amount member  → Increment score
ZRANGEBYSCORE key min max  → Query by score range
ZCARD key                  → Count members
ZPOPMIN/ZPOPMAX key        → Pop lowest/highest
```

---
## Exercises

1. **Social Network:** Create friend sets for 4 users. Find: (a) mutual friends between any two users, (b) friends of user A but not user B (friend suggestions), (c) all unique people in the network (SUNION of all).

2. **Lottery:** Add 100 numbered tickets to a set. Use SPOP to draw 5 winners. Verify no duplicate winners.

3. **Student Grades:** Create a sorted set of 10 students with exam scores. Find: (a) top 3 students, (b) students who scored above 80, (c) the rank and score of a specific student, (d) the average score.

4. **Multi-Game Leaderboard:** Create sorted sets for 3 different games. Use ZUNIONSTORE to create a combined leaderboard. Try both SUM and MAX aggregation.

5. **Scheduled Tasks:** Use a sorted set as a scheduled task queue where scores are Unix timestamps (execution time). Write a function that pops and "executes" all tasks whose scheduled time has passed.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 05 — Hashes](./05_Hashes.ipynb)** — Mini objects in Redis, perfect for user profiles, shopping carts, and configuration!